# Sentinel-2 NDVI — Field-Scale Crop Health (Microsoft Planetary Computer)

**What it is.** 10 m multispectral imagery from ESA's Sentinel-2, accessed free through the
**Microsoft Planetary Computer** STAC catalog. We compute **NDVI** — the workhorse vegetation
index — as a season-long crop-health curve for a field.

| | |
|---|---|
| **Coverage** | Global, **10 m**, ~5-day revisit (Sentinel-2 A/B) |
| **History** | 2015 → present |
| **Cost / key** | **Free** · anonymous access works; a free PC token raises limits |
| **Docs** | https://planetarycomputer.microsoft.com/dataset/sentinel-2-l2a |

**Why it's the "power" layer for Tracera.** NDVI shows *where and when* a crop is thriving or
stressed, within a field — the basis for scouting, variable-rate zones, and early problem
detection.

> **Setup.** `pip install pystac-client planetary-computer rasterio` (all in
> `requirements.txt`). No account needed to start. **Alternative backend:** Google Earth Engine
> hosts the same imagery (`COPERNICUS/S2_SR_HARMONIZED`) but needs a Google Cloud project +
> `earthengine authenticate`; Planetary Computer is lighter to script, which is why it's the
> default here.

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
import sys, pathlib, requests, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd()))
from data_sources._common import SAMPLE_FARM, get_key, field_polygon

LAT, LON = SAMPLE_FARM["lat"], SAMPLE_FARM["lon"]
FIPS, STATE, COUNTY = SAMPLE_FARM["fips"], SAMPLE_FARM["state_alpha"], SAMPLE_FARM["county_name"]
print(SAMPLE_FARM["name"], f"| lat={LAT}, lon={LON} | FIPS {FIPS}")

Story County, Iowa (sample farm) | lat=42.05, lon=-93.5 | FIPS 19169


### 1. Connection test — find low-cloud Sentinel-2 scenes over the field

In [2]:
import pystac_client, planetary_computer, rasterio
from rasterio.warp import transform as warp_transform

CATALOG = "https://planetarycomputer.microsoft.com/api/stac/v1"
catalog = pystac_client.Client.open(CATALOG, modifier=planetary_computer.sign_inplace)

def find_scenes(start, end, max_cloud=10, lat=LAT, lon=LON):
    search = catalog.search(collections=["sentinel-2-l2a"],
        intersects={"type": "Point", "coordinates": [lon, lat]},
        datetime=f"{start}/{end}", query={"eo:cloud_cover": {"lt": max_cloud}})
    return sorted(search.items(), key=lambda i: i.datetime)

scenes = find_scenes("2023-05-01", "2023-10-01")
print(f"{len(scenes)} Sentinel-2 scenes (<10% cloud) over the field, May–Oct 2023.")
print("cleanest:", min(scenes, key=lambda s: s.properties["eo:cloud_cover"]).id)

38 Sentinel-2 scenes (<10% cloud) over the field, May–Oct 2023.
cleanest: S2B_MSIL2A_20230908T165849_R069_T15TVG_20230909T011700


### 2. NDVI at the field for one scene (windowed read of just the field pixel)

In [3]:
def _pixel(href, lat=LAT, lon=LON):
    """Read a single pixel value from a remote COG at lon/lat (efficient windowed fetch)."""
    with rasterio.open(href) as ds:
        xs, ys = warp_transform("EPSG:4326", ds.crs, [lon], [lat])
        return float(next(ds.sample([(xs[0], ys[0])]))[0])

def _reflectance(dn):
    # Sentinel-2 L2A baseline >=04.00 (since 2022): BOA reflectance = (DN - 1000) / 10000
    return max((dn - 1000) / 10000, 0.0)

def ndvi_at_point(item, lat=LAT, lon=LON):
    red = _reflectance(_pixel(item.assets["B04"].href, lat, lon))   # red
    nir = _reflectance(_pixel(item.assets["B08"].href, lat, lon))   # near-infrared
    return (nir - red) / (nir + red) if (nir + red) else float("nan")

clean = min(scenes, key=lambda s: s.properties["eo:cloud_cover"])
print(f"{clean.datetime.date()}  cloud={clean.properties['eo:cloud_cover']:.2f}%  "
      f"NDVI={ndvi_at_point(clean):.3f}")

2023-09-08  cloud=0.00%  NDVI=0.356


### 3. Core — NDVI season curve at the field (crop green-up → senescence)

In [4]:
rows = []
for s in scenes[:14]:                     # cap for a quick run; widen as needed
    try:
        rows.append({"date": s.datetime.date(),
                     "cloud_%": round(s.properties["eo:cloud_cover"], 2),
                     "NDVI": round(ndvi_at_point(s), 3)})
    except Exception as e:
        print("skip", s.datetime.date(), type(e).__name__)
ndvi_ts = pd.DataFrame(rows).set_index("date")
print("Sentinel-2 NDVI time series at the field:")
ndvi_ts

Sentinel-2 NDVI time series at the field:


,cloud_%,NDVI
date,,
2023-05-01,1.27,0.212
2023-05-01,1.24,0.220
2023-05-04,0.00,0.182
2023-05-19,8.02,0.169
2023-05-24,0.00,0.217
2023-05-24,0.00,0.218
2023-05-26,0.75,0.227
2023-05-26,0.75,0.225
2023-06-15,0.19,0.440


### Notes & how to extend
- **Whole-field mean (not one pixel):** use `odc-stac`/`stackstac` to load B04 & B08 for a field
  polygon (`field_polygon()` from `_common`) and average NDVI across the field, masking clouds
  with the `SCL` band.
- **Other indices:** EVI, NDWI (water/moisture), NDRE (nitrogen) — swap the bands (B8A, B11, B05).
- **Harmonization:** the `(DN-1000)/10000` offset applies to Sentinel-2 processing baseline
  ≥ 04.00 (all 2022+ scenes); older scenes use `DN/10000`.
- **Scale up:** a PC subscription key (free) raises rate limits; for heavy use, run compute in the
  PC Hub next to the data.
- Cross-reference **CDL** (is this pixel actually corn?) and **OpenET** (water use) for the same field.